In [2]:
import torch
_ = torch.tensor([0.2126, 0.7152, 0.0722], names=['c'])

C:\Users\Administrator\AppData\Local\Temp\ipykernel_32364\97062541.py:2: UserWarning: Named tensors and all their associated APIs are an experimental feature and subject to change. Please do not use them for anything important until they are released as stable. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/core/TensorImpl.h:1938.)
  _ = torch.tensor([0.2126, 0.7152, 0.0722], names=['c'])


In [6]:
img_t = torch.randn(3, 5, 5) # shape [channels, rows, columns]
weights = torch.tensor([0.2126, 0.7152, 0.0722])

print(img_t)

tensor([[[ 0.9161, -0.5912,  0.5739, -0.2168, -0.5821],
         [ 0.1483,  0.0707, -1.1248, -1.1006, -1.8983],
         [-1.6063, -0.1265, -2.0823, -2.1221, -0.8670],
         [ 1.8681,  0.8378,  1.0494,  0.0641,  0.5025],
         [-0.7536, -0.8795, -0.4603, -0.9013, -1.0106]],

        [[-1.1202, -0.6340, -0.0583,  0.5277, -0.9233],
         [ 0.0993,  0.5814,  0.4510, -0.5627, -1.5624],
         [ 0.7448,  0.7921,  0.3626, -0.6077,  0.3048],
         [-0.0896, -0.8101, -0.8515,  1.4854, -1.2664],
         [-0.1443,  0.8253, -0.3356, -0.1665, -0.5702]],

        [[ 0.3526,  0.4475,  1.2211,  0.1216,  1.1165],
         [ 0.0730, -0.7749, -1.2928, -0.6450, -0.9261],
         [ 0.9788,  0.7521,  0.0331,  0.7995, -0.1944],
         [ 0.1045,  0.9410, -0.2447,  0.5545, -0.0968],
         [ 0.4349,  0.1905, -0.2783, -0.0887, -0.8225]]])


In [7]:
batch_t = torch.randn(2, 3, 5, 5) # shape [batch, channels, rows, columns]

In [8]:
img_gray_naive = img_t.mean(-3)
batch_gray_naive = batch_t.mean(-3)
img_gray_naive.shape, batch_gray_naive.shape

# mean 操作：所有元素的和 / 元素的个数。即求一组数值的平均值。
print(img_gray_naive)


tensor([[ 0.0495, -0.2592,  0.5789,  0.1442, -0.1296],
        [ 0.1069, -0.0409, -0.6555, -0.7695, -1.4623],
        [ 0.0391,  0.4726, -0.5622, -0.6435, -0.2522],
        [ 0.6276,  0.3229, -0.0156,  0.7013, -0.2869],
        [-0.1543,  0.0454, -0.3581, -0.3855, -0.8011]])


In [16]:
unsqueezed_weights = weights.unsqueeze(-1).unsqueeze_(-1)
print("unsqueezed_weights", unsqueezed_weights, "size", unsqueezed_weights.shape)

img_weights = (img_t * unsqueezed_weights)
img_gray_weighted = img_weights.sum(-3)

batch_weights = (batch_t * unsqueezed_weights) # 通过广播机制运算
batch_gray_weighted = batch_weights.sum(-3)

print("batch_t shape", batch_t.shape)
print("batch_weights = batch_t *  unsqueezed_weights")
print("batch_weights shape", batch_weights.shape)

print("batch_gray_weighted = batch_weights.sum(-3), batch_gray_weighted shape", batch_gray_weighted.shape)


unsqueezed_weights tensor([[[0.2126]],

        [[0.7152]],

        [[0.0722]]]) size torch.Size([3, 1, 1])
batch_t shape torch.Size([2, 3, 5, 5])
batch_weights = batch_t *  unsqueezed_weights
batch_weights shape torch.Size([2, 3, 5, 5])
batch_gray_weighted = batch_weights.sum(-3), batch_gray_weighted shape torch.Size([2, 5, 5])


In [7]:
img_gray_weighted_fancy = torch.einsum('...chw,c->...hw', img_t, weights)
batch_gray_weighted_fancy = torch.einsum('...chw,c->...hw', batch_t, weights)
batch_gray_weighted_fancy.shape

torch.Size([2, 5, 5])

In [8]:
weights_named = torch.tensor([0.2126, 0.7152, 0.0722], names=['channels'])
weights_named

tensor([0.2126, 0.7152, 0.0722], names=('channels',))

In [9]:
img_named =  img_t.refine_names(..., 'channels', 'rows', 'columns')
batch_named = batch_t.refine_names(..., 'channels', 'rows', 'columns')
print("img named:", img_named.shape, img_named.names)
print("batch named:", batch_named.shape, batch_named.names)

img named: torch.Size([3, 5, 5]) ('channels', 'rows', 'columns')
batch named: torch.Size([2, 3, 5, 5]) (None, 'channels', 'rows', 'columns')


In [10]:
weights_aligned = weights_named.align_as(img_named)
weights_aligned.shape, weights_aligned.names

(torch.Size([3, 1, 1]), ('channels', 'rows', 'columns'))

In [11]:
gray_named = (img_named * weights_aligned).sum('channels')
gray_named.shape, gray_named.names

(torch.Size([5, 5]), ('rows', 'columns'))

In [12]:
try:
    gray_named = (img_named[..., :3] * weights_named).sum('channels')
except Exception as e:
    print(e)

Error when attempting to broadcast dims ['channels', 'rows', 'columns'] and dims ['channels']: dim 'columns' and dim 'channels' are at the same position from the right but do not match.


In [13]:
gray_plain = gray_named.rename(None)
gray_plain.shape, gray_plain.names

(torch.Size([5, 5]), (None, None))